# Bắt đầu: từ trận đấu đến báo cáo

Dữ liệu thật Ngoại hạng Anh 2025–2026. SQL chạy qua Python.

**Câu hỏi:** Bảng điểm và biểu đồ được tính từ đâu?

**Dự đoán:** trước khi chạy, viết mỗi dòng kết quả đại diện cho gì.

In [1]:
from pathlib import Path
import os, sys
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'epl_lab').is_dir())
sys.path.insert(0, str(root))
from epl_lab.learning import database, read_sql
db = database()
print('SQLite chỉ đọc:', db)


SQLite chỉ đọc: C:\Users\Dell XPS\Documents\Codex\2026-10-12\h\outputs\premier-league-analytics-lab\database\premier.db


### 1. Đọc bảng nguồn
Mỗi dòng matches là một trận, không phải một đội hoặc cầu thủ.

In [2]:
read_sql('SELECT * FROM matches ORDER BY played_on LIMIT 5')

,match_id,source_id,source_row,played_on,kickoff_time,home_team_id,away_team_id,home_goals,away_goals,result,home_shots,away_shots,home_target,away_target,home_corners,away_corners,home_yellow,away_yellow,home_red,away_red
0,1,1,2,2025-08-15,20:00,12,3,4,2,H,19,10,10,3,6,7,1,2,0,0
1,2,1,3,2025-08-16,12:30,2,15,0,0,D,3,16,3,3,3,6,1,1,1,0
2,3,1,4,2025-08-16,15:00,5,10,1,1,D,10,7,4,2,4,3,3,3,0,0
3,4,1,5,2025-08-16,15:00,17,19,3,0,H,10,12,5,4,5,7,0,1,0,0
4,5,1,6,2025-08-16,15:00,18,6,3,0,H,16,14,6,4,6,5,0,0,0,0


### 2. Kiểm tra trước khi cộng
Toàn mùa phải có380trận20đội; không tự bỏ dòng sai.

In [3]:
from epl_lab.analysis import quality, queries, reconcile, scope
quality(db)

,check,value
0,matches,380
1,teams,20
2,duplicate_pair,0
3,optional_stat_missing_cells,0
4,foreign_key_errors,0


### 3. JOIN đội hai lần
Alias h/a cùng đến teams; số dòng vẫn là số trận.

In [4]:
read_sql('SELECT m.played_on,h.name home,a.name away,m.home_goals,m.away_goals FROM matches m JOIN teams h ON h.team_id=m.home_team_id JOIN teams a ON a.team_id=m.away_team_id LIMIT 10')

,played_on,home,away,home_goals,away_goals
0,2025-08-15,Liverpool,Bournemouth,4,2
1,2025-08-16,Aston Villa,Newcastle,0,0
2,2025-08-16,Brighton,Fulham,1,1
3,2025-08-16,Sunderland,West Ham,3,0
4,2025-08-16,Tottenham,Burnley,3,0
5,2025-08-16,Wolves,Man City,0,4
6,2025-08-17,Chelsea,Crystal Palace,0,0
7,2025-08-17,Nott'm Forest,Brentford,3,1
8,2025-08-17,Man United,Arsenal,0,1
9,2025-08-18,Leeds,Everton,1,0


### 4. SQL: hai góc nhìn của mỗi trận
UNION ALL giữ cả sân nhà và sân khách. COUNT đội–trận gấp đôi trận nguồn.

In [5]:
from epl_lab.analysis import team_cte
as_of=scope(db)
read_sql(team_cte()+'SELECT team_id,COUNT(*) played,SUM(points) points FROM team_matches GROUP BY team_id ORDER BY points DESC', {'as_of':as_of})

,team_id,played,points
0,1,38,85
1,13,38,78
2,14,38,71
3,2,38,65
4,12,38,60
5,3,38,57
6,17,38,54
7,5,38,53
8,4,38,53
9,10,38,52


### 5. pandas tính lại độc lập
Hàm pandas đọc bảng matches/teams rồi tự mở rộng hai phía; không đọc kết quả SQL tổng hợp.

In [6]:
tables=reconcile(db, as_of)
assert len(tables['team_matches'])==760
tables['standings']

,rank,team_id,name,played,won,drawn,lost,gf,ga,gd,points
0,1,1,Arsenal,38,26,7,5,71,27,44,85
1,2,13,Man City,38,23,9,6,77,35,42,78
2,3,14,Man United,38,20,11,7,69,50,19,71
3,4,2,Aston Villa,38,19,8,11,56,49,7,65
4,5,12,Liverpool,38,17,9,12,63,53,10,60
5,6,3,Bournemouth,38,13,18,7,58,54,4,57
6,7,17,Sunderland,38,14,12,12,42,48,-6,54
7,8,5,Brighton,38,14,11,13,52,46,6,53
8,9,4,Brentford,38,14,11,13,55,52,3,53
9,10,7,Chelsea,38,14,10,14,58,52,6,52


## Bổ sung: kiểm soát bóng trung bình
Chọn menu 7 hoặc chạy chuẩn bị với `--possession` ngoài notebook trước. Trung bình chỉ dùng trận hợp lệ đến mốc; NULL không phải 0. Đối chiếu 379 trận hợp lệ và một xung đột tỷ số; không dùng trung bình toàn mùa cho mốc nửa mùa.


In [ ]:
tables['standings'][['name','avg_possession_pct','possession_matches','possession_missing']].round({'avg_possession_pct':2})

### 6. Phong độ và mốc lịch sử
Chỉ lấy trận đến hết ngày; tối đa5trận/đội, luôn hiển thị số trận.

In [ ]:
queries(db,'2026-01-01')['last5_form']

### 7. Vẽ bốn hình và xuất bảng
Đã đối chiếu xong mới xuất; không thay đổi database.

In [ ]:
from epl_lab.analyze import analyze
out=analyze(db,as_of,root/'outputs/notebook')
from IPython.display import display, Image
for path in sorted(out.glob('*.png')): display(Image(filename=str(path)))

### 8. Đọc ba nhận xét và giới hạn
Điểm từ tỷ số, không phải mọi quyết định hành chính; tương quan không phải nguyên nhân.

In [ ]:
print((out/'report.md').read_text(encoding='utf-8'))

## Tự làm và kiểm tra

Chọn một đội bạn thích, tự kiểm tra GF/GA/điểm từ các trận. Viết3nhận xét gồm số liệu, mốc và giới hạn. Thử thay mốc rồi quan sát thay đổi.

Chỉ mở `docs/solutions.md` sau khi thử. Không sửa dữ liệu thật; bài ghi dùng database tạm.